# Phase 4: MCP (Model Context Protocol)

MCP is a standard way to offer tools from a separate server. Any MCP-capable app can use them, with no custom glue code.

Our server (`mcp_server/server.py`) offers three tools. The Routing Agent connects to it and uses them.

**Before you run the cells:** open a new terminal in VS Code, run the command below, and leave it running.

```bash
uv run python -m community_voice.mcp_server.server
```

In [ ]:
from community_voice.agents.routing import arun_routing_agent
from community_voice.agents.triage import triage
from community_voice.mcp_server.client import get_mcp_tools
from community_voice.schemas.report import ReportIn

## 1. Tools from the server

The client connects to the running server and asks what it offers. Names and descriptions come from the server's code.

In [ ]:
tools = await get_mcp_tools()
for tool in tools:
    print(tool.name, "-", tool.description.splitlines()[0])

## 2. Call a tool through MCP

In [ ]:
find = next(tool for tool in tools if tool.name == "find_municipality")
print(await find.ainvoke({"area": "Umlazi V Section"}))

## 3. Route a report with MCP tools

The same Routing Agent as Phase 2. Only the source of its tools changed.

In [ ]:
def show_trace(state):
    for message in state["messages"]:
        if message.type == "ai" and message.tool_calls:
            for call in message.tool_calls:
                if call["name"] != "RoutingResult":
                    print("ACT   ", call["name"], call["args"])
        elif message.type == "tool" and message.name != "RoutingResult":
            print("RESULT", message.text)

In [ ]:
report = ReportIn(
    text="Amanzi ayaphuma emgwaqeni e-Umlazi V Section, "
    "kuvuza ipayipi elikhulu kusukela izolo."
)
triaged = triage(report)
state = await arun_routing_agent(triaged, tools)
show_trace(state)
print(state["structured_response"].model_dump_json(indent=2))

## What to notice

- The tools live in a separate server process. The agent only knows their names and descriptions.
- The agent code did not change, only where its tools come from.
- Any other MCP client could use this server too.
- Stop the server with Ctrl+C in its terminal when you are done.